In [1]:
import json
import html
import re
import pandas as pd
from collections import defaultdict
from pathlib import Path
from utils import fix_mojibake


# -------------------------
# INPUT / OUTPUT
# -------------------------
FUZZY_JSONL = "Dataset/VIOLIN_12-10-2025/interim/fuzzy_matches/matched_fuzzy_abstracts.jsonl"
ABSTRACTS_JSONL = "Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl"
LEXICON_CSV = "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.csv"
OUTPUT_HTML = "Dataset/VIOLIN_12-10-2025/interim/fuzzy_matches/green_highlight_preview_fuzzy.html"

# -------------------------
# PMID -> title/abstract map
# -------------------------
pmid_to_abs = {}
with open(ABSTRACTS_JSONL, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pmid = str(rec.get("pmid", "")).strip()
        if pmid:
            pmid_to_abs[pmid] = {
                "title": rec.get("title", ""),
                "abstract": rec.get("abstract", "")
            }

# -------------------------
# Load lexicon map (VO -> preferred + synonyms)
# -------------------------
lex = pd.read_csv(LEXICON_CSV, encoding="utf-8-sig")

def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                import ast
                v = ast.literal_eval(s)
                return v if isinstance(v, list) else []
            except Exception:
                return []
        return [s] if s else []
    return []

lex["synonyms"] = lex["synonyms"].apply(to_list)
lexicon_map = {
    row["adjuvant_vo_id"]: {
        "preferred": row.get("preferred_name", ""),
        "synonyms": row.get("synonyms", [])
    }
    for _, row in lex.iterrows()
}

# -------------------------
# HTML helpers
# -------------------------
HTML_HEADER = """
<html>
<head>
<meta charset="utf-8"/>
<style>
body { font-family: Arial, sans-serif; line-height: 1.6; margin: 30px; }
.abstract { margin-bottom: 40px; }
.meta { color: #555; font-size: 0.9em; margin-bottom: 6px; }
</style>
</head>
<body>
<h1>Adjuvant Dictionary Matches (Fuzzy)</h1>
"""
HTML_FOOTER = "</body></html>"

def gradient_for_level(level: int, max_level: int) -> str:
    if max_level <= 1:
        return "linear-gradient(90deg, #d8fbd8, #7cf27c)"
    t = (level - 1) / (max_level - 1)
    r = int(216 + (72 - 216) * t)
    g = int(251 + (186 - 251) * t)
    b = int(216 + (96 - 216) * t)
    return f"linear-gradient(90deg, rgb({r},{g},{b}), rgb({max(0,r-20)},{max(0,g-40)},{max(0,b-40)}))"

def highlight_text_gradient(text, spans):
    if not spans:
        return html.escape(text)
    n = len(text)
    depth = [0] * n
    for start, end in spans:
        start = max(0, min(n, start))
        end = max(0, min(n, end))
        for i in range(start, end):
            depth[i] += 1
    max_level = max(depth) if depth else 0

    out = []
    i = 0
    while i < n:
        if depth[i] == 0:
            j = i + 1
            while j < n and depth[j] == 0:
                j += 1
            out.append(html.escape(text[i:j]))
            i = j
        else:
            level = depth[i]
            j = i + 1
            while j < n and depth[j] == level:
                j += 1
            color = gradient_for_level(level, max_level)
            out.append(
                f"<span style='background:{color}; padding:1px 2px; border-radius:3px;'>"
                f"{html.escape(text[i:j])}</span>"
            )
            i = j
    return "".join(out)

def build_spans(text, terms):
    spans = []
    if not text:
        return spans
    lower = text.lower()
    for term in terms:
        if not isinstance(term, str):
            continue
        t = term.strip().lower()
        if len(t) < 3:
            continue
        pat = re.escape(t)
        pat = pat.replace(r"\ ", r"\s+")
        pat = pat.replace(r"\-", r"[-\s]+")
        regex = re.compile(rf"\b{pat}\b", flags=re.I)
        for m in regex.finditer(lower):
            spans.append((m.start(), m.end()))
    return spans

# -------------------------
# Build HTML
# -------------------------
html_blocks = [HTML_HEADER]

with open(FUZZY_JSONL, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        pmid = str(record.get("pmid", "")).strip()

        abs_rec = pmid_to_abs.get(pmid, {"title": "", "abstract": ""})
        #title = abs_rec.get("title", "")
        #abstract = abs_rec.get("abstract", "")
        title = fix_mojibake(abs_rec.get("title", ""))
        abstract = fix_mojibake(abs_rec.get("abstract", ""))


        matches = record.get("matches", [])
        if not abstract or not matches:
            continue

        # Build VO list using preferred + synonyms
        vo_lines = []
        for vo in sorted({m["adjuvant_vo_id"] for m in matches}):
            info = lexicon_map.get(vo, {})
            #pref = info.get("preferred", "")
            #syns = info.get("synonyms", [])
            #syns_short = syns[:10]
            #vo_lines.append(f"{vo}: {pref} | synonyms: {', '.join(syns_short)}")
            pref = fix_mojibake(info.get("preferred", ""))
            syns = info.get("synonyms", [])
            syns_short = [fix_mojibake(s) for s in syns[:10]]
            vo_lines.append(f"{vo}: {pref} | synonyms: {', '.join(syns_short)}")


        # Build terms for highlighting from matched_text
        terms = set()
        for m in matches:
            t = m.get("matched_text", "")
            if t:
                terms.add(t)

        spans = build_spans(abstract, terms)
        highlighted = highlight_text_gradient(abstract, spans)

        html_blocks.append(
            f"""
            <div class="abstract">
                <div class="meta">
                    <b>PMID:</b> {pmid}<br/>
                    <b>Title:</b> {html.escape(title)}<br/>
                    <b>Mentions:</b> {len(matches)} |
                    <b>Unique adjuvants:</b> {len(set(m["adjuvant_vo_id"] for m in matches))}<br/>
                    <b>Adjuvants (by VO):</b> {" | ".join(vo_lines)}<br/>
                </div>
                <div>{highlighted}</div>
            </div>
            <hr/>
            """
        )

html_blocks.append(HTML_FOOTER)

Path(OUTPUT_HTML).parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_HTML, "w", encoding="utf-8") as out:
    out.write("\n".join(html_blocks))

print(f"Saved: {OUTPUT_HTML}")


Saved: Dataset/VIOLIN_12-10-2025/interim/fuzzy_matches/green_highlight_preview_fuzzy.html
